In [1]:
import os
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ModuleNotFoundError:
    ! pip install seaborn
    import seaborn as sns

#### Functions

In [ ]:
# plot data types
def plot_data_types(df_data, str_filename='output/data_types.png'):
    # get value counts of dtypes
    ser_dtype_counts = df_data.dtypes.astype(str).value_counts()
    int_total = ser_dtype_counts.sum()
    # plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ser_dtype_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black', rot=0)
    ax.set_title('Data Types', fontsize=16)
    ax.set_xlabel('Data Type', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    # annotate bars with count and percent
    for i, int_val in enumerate(ser_dtype_counts):
        flt_pct = int_val / int_total * 100
        ax.text(i, int_val + (ax.get_ylim()[1] * 0.01), f'{int_val}\n({flt_pct:.1f}%)', ha='center', fontsize=11)
    # pad the top so labels don't clip
    ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

In [3]:
# plot proportion missing
def plot_proportion_missing(df_data, str_filename='output/proportion_missing.png'):
    # calculate proportion missing
    ser_missing = df_data.isnull().mean()
    # filter to columns with missing values
    ser_missing = ser_missing[ser_missing > 0].sort_values(ascending=True)
    # if no missing values
    if len(ser_missing) == 0:
        print('No missing values found.')
        return
    # plot
    fig, ax = plt.subplots(figsize=(10, max(5, len(ser_missing) * 0.4)))
    ser_missing.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Proportion Missing by Column', fontsize=16)
    ax.set_xlabel('Proportion Missing', fontsize=12)
    ax.set_ylabel('Column', fontsize=12)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

In [ ]:
# plot target distribution
def plot_target(df_data, str_target, str_filename='output/target_distribution.png'):
    # value counts
    ser_counts = df_data[str_target].value_counts()
    ser_proportions = df_data[str_target].value_counts(normalize=True)
    # plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ser_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black', rot=0)
    ax.set_title('Target Distribution', fontsize=16)
    ax.set_xlabel(str_target, fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    # annotate bars with count and proportion
    for i, (int_count, flt_prop) in enumerate(zip(ser_counts, ser_proportions)):
        ax.text(i, int_count + (ax.get_ylim()[1] * 0.01), f'{int_count:,}\n({flt_prop:.2%})', ha='center', fontsize=11)
    # pad the top so labels don't clip
    ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

In [ ]:
# descriptive statistics
def descriptive_statistics(df_data):
    df_desc = df_data.describe().T.reset_index()
    df_desc = df_desc.rename(columns={'index': 'str_column'})
    # add unique count
    df_desc['int_n_unique'] = df_desc['str_column'].apply(lambda x: df_data[x].nunique())
    # add proportion missing
    df_desc['flt_proportion_missing'] = df_desc['str_column'].apply(lambda x: df_data[x].isnull().mean())
    return df_desc

In [ ]:
# plot violin plots
def plot_violin(df_data, str_target, str_filename='output/violin_plots.png'):
    import warnings
    warnings.filterwarnings('ignore')
    # get numeric columns excluding target
    list_numeric_cols = [col for col in df_data.select_dtypes(include=[np.number]).columns if col != str_target]
    int_n_cols = 3
    int_n_rows = int(np.ceil(len(list_numeric_cols) / int_n_cols))
    fig, axes = plt.subplots(int_n_rows, int_n_cols, figsize=(6 * int_n_cols, 4 * int_n_rows))
    axes = axes.flatten() if int_n_rows > 1 else (axes if hasattr(axes, '__iter__') else [axes])
    for i, str_col in enumerate(list_numeric_cols):
        sns.violinplot(data=df_data, x=str_target, y=str_col, ax=axes[i], hue=str_target, palette='muted', legend=False)
        axes[i].set_title(str_col, fontsize=12)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')
    # hide unused axes
    for j in range(len(list_numeric_cols), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Violin Plots by Target', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300, bbox_inches='tight')
    plt.show()
    warnings.filterwarnings('default')

In [7]:
# plot correlation with target
def plot_correlation_with_target(df_data, str_target, str_filename='output/correlation_with_target.png'):
    # compute correlations with target
    ser_corr = df_data.select_dtypes(include=[np.number]).corr()[str_target].drop(str_target)
    # sort by absolute value
    ser_corr = ser_corr.reindex(ser_corr.abs().sort_values(ascending=True).index)
    # plot
    fig, ax = plt.subplots(figsize=(10, max(5, len(ser_corr) * 0.4)))
    list_colors = ['steelblue' if x >= 0 else 'salmon' for x in ser_corr]
    ser_corr.plot(kind='barh', ax=ax, color=list_colors, edgecolor='black')
    ax.set_title('Correlation with Target', fontsize=16)
    ax.set_xlabel('Correlation', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    ax.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

#### Constants

In [8]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# s3 path
str_s3_path = f's3://{str_bucket}/00_data_collection/data.csv'
print(f'S3 Path: {str_s3_path}')

# target
str_target = 'default_12m'
print(f'Target: {str_target}')

# output directory
os.makedirs('output', exist_ok=True)

Bucket: credit-risk-claude
Step: 01_eda
S3 Path: s3://credit-risk-claude/00_data_collection/data.csv
Target: default_12m


#### Read Data

In [ ]:
# read data
df = pd.read_csv(str_s3_path)

# rows and columns
int_n_rows = df.shape[0]
int_n_cols = df.shape[1]
print(f'Rows: {int_n_rows:,}')
print(f'Columns: {int_n_cols}')

# date range (if date column exists)
list_date_cols = df.select_dtypes(include=['datetime64']).columns.tolist()
if len(list_date_cols) == 0:
    # try to find date-like object columns
    for str_col in df.select_dtypes(include=['object']).columns:
        try:
            pd.to_datetime(df[str_col].head(100))
            list_date_cols.append(str_col)
        except (ValueError, TypeError):
            pass
if len(list_date_cols) > 0:
    str_date_col = list_date_cols[0]
    df[str_date_col] = pd.to_datetime(df[str_date_col])
    print(f'Date Column: {str_date_col}')
    print(f'Date Range: {df[str_date_col].min()} to {df[str_date_col].max()}')

# target mean
flt_target_mean = df[str_target].mean()
print(f'Target Mean: {flt_target_mean:.4f}')

# preview
df.head()

#### Data Types

In [ ]:
plot_data_types(df)

#### Missing Values

In [ ]:
plot_proportion_missing(df)

#### Target Variable

In [ ]:
plot_target(df, str_target)

#### Descriptive Statistics

In [ ]:
df_desc = descriptive_statistics(df)
df_desc.to_csv('output/descriptive_statistics.csv', index=False)
df_desc

#### Violin Plots

In [ ]:
plot_violin(df, str_target)

#### Correlation with Target

In [ ]:
plot_correlation_with_target(df, str_target)